In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'

# Load master table
master = pd.read_csv(DATA_PROCESSED / 'master.csv')
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
master['order_delivered_customer_date'] = pd.to_datetime(master['order_delivered_customer_date'])
master['order_estimated_delivery_date'] = pd.to_datetime(master['order_estimated_delivery_date'])

# Load reviews separately — it's not in master
reviews = pd.read_csv(DATA_RAW / 'olist_order_reviews_dataset.csv')

print(f"master:  {master.shape[0]:,} rows")
print(f"reviews: {reviews.shape[0]:,} rows")

master:  110,197 rows
reviews: 99,224 rows


In [3]:
#Get first purchase month for each customer
cohort_data=master[['customer_unique_id','order_purchase_timestamp','order_id']].copy()
cohort_data['order_month']=cohort_data['order_purchase_timestamp'].dt.to_period('M')

#Cohort month= monthh of first purchase
cohort_data['cohort_month']=cohort_data.groupby('customer_unique_id')['order_month'].transform('min')


#Month index=number of months since first purchase
cohort_data['month_index']=(
    (cohort_data['order_month']-cohort_data['cohort_month']).apply(lambda x:x.n)
)


#Count unique customers per cohort per month
cohort_pivot=(cohort_data
              .groupby(['cohort_month','month_index'])['customer_unique_id']
              .nunique()
              .unstack(fill_value=0))

#Rentention rate relative to cohort size
cohort_size=cohort_pivot[0]
retention=cohort_pivot.divide(cohort_size,axis=0)*100

retention.index=retention.index.astype(str)
print(retention.head())

month_index      0           1         2        3         4        5   \
cohort_month                                                            
2016-09       100.0    0.000000  0.000000  0.00000  0.000000  0.00000   
2016-10       100.0    0.000000  0.000000  0.00000  0.000000  0.00000   
2016-12       100.0  100.000000  0.000000  0.00000  0.000000  0.00000   
2017-01       100.0    0.278940  0.278940  0.13947  0.418410  0.13947   
2017-02       100.0    0.184275  0.307125  0.12285  0.429975  0.12285   

month_index         6         7        8         9        10        11  \
cohort_month                                                             
2016-09       0.000000  0.000000  0.00000  0.000000  0.00000  0.000000   
2016-10       0.381679  0.000000  0.00000  0.381679  0.00000  0.381679   
2016-12       0.000000  0.000000  0.00000  0.000000  0.00000  0.000000   
2017-01       0.418410  0.139470  0.13947  0.000000  0.41841  0.139470   
2017-02       0.245700  0.184275  0.12285  0

In [ ]:
import matplotlb.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 10))

sns.heatmap(
    retention.iloc[:,13],
    annot=True,
    fmt='.0f',
    cmap='Blues',
    vmin=0,vmax=15,
    linewidths=0.5,
    cbar_kws={'label': 'Retention rate (%)'},
)

plt.title('Customer Cohort Retention Rate (%)',fontsize=16, pad=20)
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'cohort_retention_heatmap.png',dpi=150,bbox_inches='tight')